In [20]:
import sys
import os
import numpy as np
from scipy import stats
import time
import pandas as pd

sys.path.append(os.path.abspath('..'))

from LSM.stochastic_processes import GeometricBrownianMotion
from LSM.payoffs import VanillaPayoff
from LSM.regression_bases import LaguerrePolynomials
from LSM.algorithms import LeastSquaresMonteCarlo
from LSM.binomial_tree import BinomialTreeEngine
from LSM.control_variate import *

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
#(1) BSM Benchmark: Black-Scholes-Merton Model for European Options
def BSM(S0: float, K: float, T: float, r: float, q: float, sigma: float, option_type: str) -> float:
    """
    Returns the BSM price estimation of a European Option
    """
    option_type = option_type.lower()
    if option_type not in ['call', 'put']:
        raise ValueError("option_type must be either 'call' or 'put'.")

    d1 = (np.log(S0/K) + (r - q + 0.5*(sigma**2))*T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)

    if option_type == 'put':
        price = np.exp(-r*T) * K * stats.norm.cdf(-d2) - np.exp(-q*T) * S0 * stats.norm.cdf(-d1)
    else:
        price = np.exp(-q*T) * S0 * stats.norm.cdf(d1) - np.exp(-r*T) * K * stats.norm.cdf(d2)
    
    std = None

    return price, std

In [22]:
# Sanity check: an American call without dividend (q=0) is equivalent to a 
# European call. Compare our LSM price with the BSM European call price.
# Accuracy, runtime, (memory storage)

# Sample test parameters
S0 = 36
K = 40
r = 0.06
q = 0.0
sigma = 0.2
T = 1.0
n_steps = 50
n_paths = 10000

# BSM Sanity Check: (American Call Option = European Call)
start_bsm = time.time()
bsm_call = BSM(S0, K, T, r, q, sigma, option_type = "call")[0]
end_bsm = time.time()
print(f"BSM European Call Price: {bsm_call}.")

# American Call option price from Binomial Tree
start_BioTree = time.time()
call_payoff = VanillaPayoff(strike=K, option_type="call")
Binomial_eng = BinomialTreeEngine(call_payoff)
Binomial_call = Binomial_eng.pricer(S0=S0, r=r, q=q, sigma=sigma, T=T, n_steps=n_steps, american=True)
end_BioTree = time.time()
print(f"BinomialTree American Call Price {Binomial_call}")

gbm = GeometricBrownianMotion(S0=S0, r=r, q=q, sigma=sigma)
laguerre_basis = LaguerrePolynomials(degree=3)

lsm_eng = LeastSquaresMonteCarlo(
    process = gbm,
    payoff_function = call_payoff,
    basis_function=laguerre_basis
    )
start_lsm = time.time()
lsm_call = lsm_eng.pricer(T = T, n_steps = n_steps, n_paths = n_paths)[0]
end_lsm = time.time()
print(f"LSM American Call Price {lsm_call}.")
print(f"Absolute Error between BSM and LSM on Call: {abs(bsm_call - lsm_call)}")

print(f"BSM Runtime: {end_bsm - start_bsm}")
print(f"BinomialTree Runtime: {end_BioTree - start_BioTree}")
print(f"LSM Runtime: {end_lsm - start_lsm}")

BSM European Call Price: 2.1737264482268923.
BinomialTree American Call Price 2.1690295200457497
LSM American Call Price 2.1645160191876194.
Absolute Error between BSM and LSM on Call: 0.009210429039272938
BSM Runtime: 0.0005424022674560547
BinomialTree Runtime: 0.0005466938018798828
LSM Runtime: 0.0902092456817627


In [23]:
# TODO: Use the American put numerical example in Longstaff, Schartz (2001) pp. 126-128.
# Goals: 
# a) replicate their results; 
# b) compare with other benchmarks and libraries (e.g. CRR Binomial Tree),
# focusing on accuracy (prices), speed (%timeit) and memory usage
# [CREATE TESTS USING OTHER LIBRARIES FIRST!]


S0 = 36.0
r = 0.06
q = 0.0
sigma = 0.2
K = 40.0
T = 1.0
n_steps = 50
n_paths = 10000


gbm = GeometricBrownianMotion(S0=S0, r=r, q=q, sigma=sigma)
put_payoff = VanillaPayoff(strike=K, option_type="put") # "put"
laguerre_basis = LaguerrePolynomials(degree=3)


lsm_engine = LeastSquaresMonteCarlo(
    process=gbm, 
    payoff_function=put_payoff, 
    basis_function=laguerre_basis
)


time_grid, paths = gbm.simulate(T=T, n_steps=n_steps, n_paths=n_paths)
print(f"1. GBM paths shape: {paths.shape}  --> Expect ({n_paths}, {n_steps + 1})")

payoff_at_maturity = put_payoff(paths[:, -1])
print(f"2. Payoff shape: {payoff_at_maturity.shape}  --> Expect: ({n_paths},)")

test_X = paths[:100, -1]
test_Y = payoff_at_maturity[:100]
beta = laguerre_basis.fit(X=test_X, Y=test_Y)
print(f"3. beta fitted on Laguerre basis \n{beta}  --> Expect: ")


price = lsm_engine.pricer(T=T, n_steps=n_steps, n_paths=n_paths)
print(f"4. LSM price: {price}  --> Expect: ")

1. GBM paths shape: (10000, 51)  --> Expect (10000, 51)
2. Payoff shape: (10000,)  --> Expect: (10000,)
3. beta fitted on Laguerre basis 
[7.11807746e+01 3.16973273e+00 8.27788814e-02 6.97501670e-04]  --> Expect: 
4. LSM price: (4.465836305839184, 0.0019824278833265765)  --> Expect: 


In [24]:
def LSM(S0, K, T, r, q, sigma, n_steps, n_paths, degree, option_type='put', control_type=None):
    gbm = GeometricBrownianMotion(S0=S0, r=r, q=q, sigma=sigma)
    put_payoff = VanillaPayoff(strike=K, option_type=option_type)
    laguerre_basis = LaguerrePolynomials(degree=degree)


    lsm_engine = LeastSquaresMonteCarlo(
        process=gbm, 
        payoff_function=put_payoff, 
        basis_function=laguerre_basis
    )

    
    price, std = lsm_engine.pricer(
        T=T,
        n_steps=n_steps,
        n_paths=n_paths,
        rng=np.random.default_rng(42),
        use_antithetic=False,
        control_variate=control_type,
        cache=False
    )

    return price,std

In [25]:

def LSM_CV(S0, K, T, r, q, sigma, n_steps, n_paths, degree, 
           option_type='put', 
           control_type = 'european_at_maturity'):
    gbm = GeometricBrownianMotion(S0=S0, r=r, q=q, sigma=sigma)
    put_payoff = VanillaPayoff(strike=K, option_type=option_type)
    laguerre_basis = LaguerrePolynomials(degree=degree)
    
    lsm_engine = LeastSquaresMonteCarlo(
        process=gbm, 
        payoff_function=put_payoff, 
        basis_function=laguerre_basis)

    price_cv, std_cv = lsm_engine.pricer(
        T=T,
        n_steps=n_steps,
        n_paths=n_paths,
        rng=np.random.default_rng(42),
        use_antithetic=False,
        control_variate=control_type,
        cache=False
    )

    return price_cv, std_cv

In [26]:
# Test Example Binomial Tree Benchmark:
def BTM(S0, K, T, r, q, sigma, n_steps, option_type = 'put'):
    payoff = VanillaPayoff(strike=K, option_type=option_type)
    Binomial_eng = BinomialTreeEngine(payoff)
    Binomial_price = Binomial_eng.pricer(S0=S0, r=r, q=q, sigma=sigma, T=T, n_steps=n_steps, american=True)
    std = None
    return Binomial_price, std

In [27]:
# Test Example Finite Difference Benchmark: 
def FDM(S0, K, T, r, q, sigma, n_steps, n_paths, option_type = "put"):
    std = None
    print("FDM pricer is not implemented yet.")
    return 6, std


In [28]:
#(4) Comparison Table for LSM Accuracy and RunTime （需要验证）
# Assume here it is always American Put Option
# The Benchmark must be the estimated price by a highly stable and accurate model,
    #i.e., Binominal Tree Method

def comparison(model, name: str, benchmark_price, **kwargs):
    """
    Take all existed models (BSM, LSM, ...) to compute prices
    Compare Accuracy and RunTime of our LSM with other models
    Return a Comparison Table
    """

    start = time.time()
    price,std = model(**kwargs)
    end = time.time()

    runtime = end - start
    abs_error = abs(price - benchmark_price)
    #rel_error = abs_error / benchmark_price

    Ctable = {
        "Model": name,
        "Price": price,
        "std": std,
        "Absolute Error": abs_error,
        "RunTime": runtime
    }

    return Ctable
 


In [ ]:
#Parameters 
cargs = dict(S0=S0, K=K, T=T, r=r, q=q, sigma=sigma, option_type = 'put')

#Benchmark
benchmark_price, _ = BTM(S0=S0, K=K, T=T, r=r, q=q, sigma=sigma, n_steps=n_steps, option_type='put')
print(benchmark_price)

#Realization of Comparison Table
models = []

models.append(
    comparison(LSM, "Least Square Monte Carlo", benchmark_price, **cargs, n_steps=n_steps, n_paths=n_paths, degree=3,
               control_type = None))
models.append(
    comparison(LSM_CV, "LSM + Control Variate at maturity", benchmark_price, **cargs, n_steps=n_steps, n_paths=n_paths, degree=3, 
               control_type = 'european_at_maturity'))
models.append(
    comparison(LSM_CV, "LSM + Control Variate at excercise", benchmark_price, **cargs, n_steps=n_steps, n_paths=n_paths, degree=3, 
               control_type = 'european_at_exercise'))
models.append(
    comparison(BTM, "Binomial Tree Model", benchmark_price, **cargs, n_steps=n_steps))
models.append(
    comparison(FDM, "Finite Difference Method", benchmark_price, **cargs, n_steps=n_steps, n_paths=n_paths))
models.append(
    comparison(BSM, "Black Scholes Model", benchmark_price, **cargs)
)

#Convert into a Data Frame
df = pd.DataFrame(models)

print(df)

4.4844980628541835
FDM pricer is not implemented yet.
                                Model     Price       std  Absolute Error  \
0            Least Square Monte Carlo  4.491576  0.029910        0.007078   
1   LSM + Control Variate at maturity  4.486759  0.024271        0.002261   
2  LSM + Control Variate at excercise  4.471446  0.002139        0.013052   
3                 Binomial Tree Model  4.484498       NaN        0.000000   
4            Finite Difference Method  6.000000       NaN        1.515502   
5                 Black Scholes Model  3.844308       NaN        0.640190   

    RunTime  
0  0.052333  
1  0.055188  
2  0.633395  
3  0.001252  
4  0.000000  
5  0.000000  
